# TCF7 / TCF-1 Mutation Search – Demo Notebook

This notebook demonstrates how to use `tcf1_mutation_search.py` to find
mutations in the TCF-1 protein (encoded by **TCF7**) within the query region:

```
ATGTACAAAGAGACCGTCTACTCCGCCTTCAATCTGCTCATGCATTACCC
ACCCCCCTCGGGAGCAGGGCAGCACCCCCAGCCGCAGCCCCCG
```

This corresponds to the **N-terminal** coding region of TCF7, which encodes
a portion of the DNA-binding / dimerisation domain of TCF-1.

## Install dependencies
```bash
pip install requests biopython pandas tqdm
```

In [ ]:
import importlib, sys
# Install if missing
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'biopython', 'pandas', 'tqdm'])

## 1. Align query sequence to TCF7 CDS and resolve genomic coordinates

In [ ]:
import sys
sys.path.insert(0, '.')

from tcf1_mutation_search import (
    QUERY_SEQUENCE, TCF7_TRANSCRIPT, ENSEMBL_REST,
    map_sequence_to_genome, fetch_gnomad_variants,
    classify_impact, print_summary
)

print('Query sequence:', QUERY_SEQUENCE)
print('Length (bp):   ', len(QUERY_SEQUENCE))

In [ ]:
# Map query sequence to GRCh38 genome coordinates via Ensembl
coords = map_sequence_to_genome(QUERY_SEQUENCE)
print(coords)

## 2. Fetch variants from gnomAD (population WES + WGS)

In [ ]:
df_gnomad = fetch_gnomad_variants(
    coords['chrom'], coords['start'], coords['end']
)
df_gnomad.head()

In [ ]:
# Classify impact and filter
if not df_gnomad.empty:
    df_gnomad['impact'] = df_gnomad['consequence'].fillna('').apply(classify_impact)
    print('Impact breakdown:')
    print(df_gnomad['impact'].value_counts())
    print()
    print('High/Moderate impact variants:')
    cols = ['variant_id','pos','ref','alt','consequence','hgvsp',
            'exome_af','genome_af','clinvar_clinsig']
    cols = [c for c in cols if c in df_gnomad.columns]
    hi = df_gnomad[df_gnomad['impact'].isin(['HIGH','MODERATE'])]
    display(hi[cols])

## 3. Filter for clinically significant variants

In [ ]:
if not df_gnomad.empty and 'clinvar_clinsig' in df_gnomad.columns:
    patho = df_gnomad[
        df_gnomad['clinvar_clinsig'].str.contains(
            'pathogenic|likely_pathogenic', case=False, na=False
        )
    ]
    print(f'Pathogenic / Likely Pathogenic variants: {len(patho)}')
    display(patho)

## 4. (Optional) Scan a local VCF file

In [ ]:
# Uncomment and set your VCF path:
# from tcf1_mutation_search import parse_vcf
# vcf_path = '/path/to/your/exome.vcf.gz'
# df_vcf = parse_vcf(vcf_path, coords['chrom'], coords['start'], coords['end'])
# display(df_vcf)

## 5. Save results

In [ ]:
if not df_gnomad.empty:
    df_gnomad.to_csv('tcf7_gnomad_variants.csv', index=False)
    print('Saved: tcf7_gnomad_variants.csv')